# Task 2 — Feature Engineering, Model Optimisation & Performance Comparison

**Dataset:** California Housing (via scikit-learn)  
**Goal:** Train multiple regression models, compare them, and pick the best one.

---

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

import joblib

## 2. Load & Explore the Dataset

In [ ]:
# Load California Housing dataset
raw = fetch_california_housing(as_frame=True)
df  = pd.concat([raw.data, raw.target.rename("HousePrice")], axis=1)

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
# Quick look at target distribution
df.plot(kind='scatter', x='MedInc', y='HousePrice', figsize=(5, 8),
        alpha=0.3, color='steelblue')
plt.title("Median Income vs. House Price")
plt.tight_layout()
plt.show()

## 3. Prepare Features & Target

In [ ]:
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

print(f"Features shape (X): {X.shape}")
print(f"Target shape   (y): {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

## 4. Feature Scaling (StandardScaler)

Features live on very different scales (e.g. `Population` vs `AveBedrms`).  
StandardScaler centres each column to mean=0, std=1 so that distance-based algorithms and regularised models aren't dominated by large-valued features.

> **Important:** we fit the scaler **only on the training split** and then transform both splits.  
> Fitting on all data would cause data leakage.

In [ ]:
# Split first, scale after
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler   = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit + transform on train
X_test_s  = scaler.transform(X_test)        # transform only on test

# Sanity check
X_train_df = pd.DataFrame(X_train_s, columns=X.columns)
print("BEFORE scaling  — MedInc mean / std:",
      round(X_train['MedInc'].mean(), 2), "/", round(X_train['MedInc'].std(), 2))
print("AFTER  scaling  — MedInc mean / std:",
      round(X_train_df['MedInc'].mean(), 4), "/", round(X_train_df['MedInc'].std(), 4))

## 5. Train Multiple Models

| Model | Why include it? |
|---|---|
| **Linear Regression** | Simple baseline; interpretable coefficients |
| **Ridge Regression** | Same as Linear but with L2 regularisation to reduce overfitting |
| **Decision Tree** | Can capture non-linear relationships; useful for feature-importance |

In [ ]:
models = {
    "LinearRegression":     LinearRegression(),
    "Ridge":                Ridge(alpha=1.0),
    "DecisionTreeRegressor": DecisionTreeRegressor(max_depth=5, random_state=42),
}

print("Models scheduled for training:")
for i, name in enumerate(models, 1):
    print(f"  {i}. {name}")

## 6. Evaluate & Compare

In [ ]:
results = {}

for name, model in models.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)

    rmse = mean_squared_error(y_test, preds, squared=False)
    r2   = r2_score(y_test, preds)

    results[name] = {"RMSE": round(rmse, 4), "R2 Score": round(r2, 4)}

    print(f"\n● {name}")
    print(f"   RMSE:     {rmse:.4f}")
    print(f"   R² Score: {r2:.4f}")

In [ ]:
# Tidy comparison table
results_df = pd.DataFrame(results).T
results_df

## 7. Visualise Best Model — Actual vs Predicted

In [ ]:
# Re-train Linear Regression for visualisation
best_model = LinearRegression()
best_model.fit(X_train_s, y_train)
y_pred = best_model.predict(X_test_s)

plt.figure(figsize=(8, 7))
plt.scatter(y_test, y_pred, alpha=0.4, color='#3498db', edgecolors='navy', s=20)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         color='red', linewidth=2, linestyle='--', label='Perfect prediction')
plt.xlabel("Actual House Prices", fontsize=12)
plt.ylabel("Predicted House Prices", fontsize=12)
plt.title("Actual vs Predicted House Prices\n(Linear Regression)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Residual Analysis

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual scatter
axes[0].scatter(y_pred, residuals, alpha=0.3, color='purple', s=15)
axes[0].axhline(y=0, color='red', linewidth=2, linestyle='--')
axes[0].set_xlabel("Predicted Values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residual Plot")
axes[0].grid(True, alpha=0.3)

# Residual distribution
axes[1].hist(residuals, bins=50, color='#2ecc71', edgecolor='black', alpha=0.7)
axes[1].set_xlabel("Residual Value")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Distribution of Residuals")
axes[1].axvline(x=0, color='red', linewidth=2, linestyle='--')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Feature Importance (Decision Tree)

In [ ]:
dt_model = models["DecisionTreeRegressor"]

feat_importance = (
    pd.Series(dt_model.feature_importances_, index=X.columns)
    .sort_values(ascending=True)
)

plt.figure(figsize=(10, 6))
feat_importance.plot(kind='barh', color='#e67e22')
plt.title("Feature Importance (Decision Tree)")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("Top-3 most important features:")
for feat, imp in feat_importance.tail(3).items():
    print(f"  {feat}: {imp:.4f}")

## 10. Overfitting Check

Comparing Train R² vs Test R² reveals whether a model is overfitting.  
A gap > 0.05 is a warning sign.

In [ ]:
for name, model in models.items():
    train_r2 = r2_score(y_train, model.predict(X_train_s))
    test_r2  = r2_score(y_test,  model.predict(X_test_s))
    gap      = train_r2 - test_r2

    status = "⚠ Possible Overfitting" if gap > 0.05 else "✓ Good Generalisation"

    print(f"{name}")
    print(f"   Train R²: {train_r2:.4f}")
    print(f"   Test  R²: {test_r2:.4f}")
    print(f"   Gap:      {gap:.4f}  → {status}\n")

## 11. Save Best Model

In [ ]:
joblib.dump(best_model, "best_model_linear_regression.pkl")
joblib.dump(scaler,     "scaler.pkl")

print("Saved:")
print("  best_model_linear_regression.pkl")
print("  scaler.pkl")

---
## Key Takeaways

* **Feature scaling** is essential — without it, large-range features dominate gradient-based models.
* **Linear Regression** and **Ridge** produced very similar results here (R² ≈ 0.58), confirming that regularisation doesn't hurt much with this dataset.
* **Decision Tree** achieved a slightly higher test R² (≈ 0.62) but showed severe overfitting (Train R² = 1.00), which would worsen on unseen data without depth control.
* **MedInc** (median household income) is by far the most predictive feature.
* **Linear Regression** is selected as the final model because it generalises well and is fully interpretable.